# Unsloth Qwen3 SFT/RL Notebook


## 0. 指定 GPU

In [1]:
import os
import torch

GPU_ID = "2"  # 改成 nvidia-smi 里空闲的物理 GPU 编号

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Visible GPU count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("GPU name:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info(0)
print("Free GB:", round(free / 1024**3, 2))
print("Total GB:", round(total / 1024**3, 2))


CUDA_VISIBLE_DEVICES: 2
Visible GPU count: 1
Current CUDA device: 0
GPU name: NVIDIA GeForce RTX 3090
Free GB: 23.43
Total GB: 23.69


## 1. 实验配置


In [ ]:
from pathlib import Path
import sys

# 自动识别仓库根目录：既支持从 repo root 打开，也支持从 notebooks/ 打开。
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

# ========== file path ==========
MODEL_NAME_OR_PATH = str(REPO_ROOT / "models" / "Qwen3-4B-Instruct-2507")
TRAIN_FILE = REPO_ROOT / "data" / "split_data" / "toy_train.json"  # 原始训练集 JSON 文件
TEST_FILE = REPO_ROOT / "data" / "split_data" / "toy_train.json"    # 训练集JSON 文件 测试训练效果
TEST_FILE_TEST = REPO_ROOT / "data" / "split_data" / "toy_test.json"   # 测试集 JSON 文件 
PROCESSED_TRAIN_FILE = REPO_ROOT / "data" / "processed_data" / "processed_train_messages.json"
PROCESSED_TEST_FILE = REPO_ROOT / "data" / "processed_data" / "processed_test_messages.json"
SFT_OUTPUT_DIR = REPO_ROOT / "outputs" / "lora_files" / "qwen3_4b_unsloth_lora"
PRED_OUTPUT_FILE = REPO_ROOT / "outputs" / "predictions" / "qwen3_4b_test_predictions.json"

# 你的原始数据字段：processed JSON 只保留这些关键字段。
PROMPT_FIELD = "prompt"
RESPONSE_FIELD = "groundtruth"
RESPONSE_FIELD_CANDIDATES = ["groundtruth", "response", "answer", "label", "output", "completion"]
ID_FIELD = "prompt_id"
CLAIM_ID_FIELD = "claim_id"
CONDITION_FIELD = "condition"
PROCESSED_KEEP_FIELDS = [ID_FIELD, CLAIM_ID_FIELD, PROMPT_FIELD, RESPONSE_FIELD, CONDITION_FIELD]
SYSTEM_PROMPT = "You are a respondent in a persuasion scenario. Answer from the assigned role's perspective."
ASSISTANT_ROLE = "assistant"  # 不建议改成 respondent；多数 chat template 只支持 assistant。
ENABLE_THINKING = False      # Qwen3 默认可能插入 <think>...</think>；SFT/量表任务建议关闭。

# 模型 / QLoRA 设置
MAX_SEQ_LENGTH = 512       # 如果 OOM，先降到 512
LOAD_IN_4BIT = True         # True=QLoRA；False=普通 LoRA
DTYPE = None                # None 让 Unsloth 自动选择

# LoRA 超参
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# SFT 超参
RESPONSE_ONLY_LOSS = True   # True=只训练 groundtruth/assistant answer；False=prompt+answer 都算 loss
NUM_TRAIN_EPOCHS = 8
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
GRAD_ACCUM = 4

# 批量推理参数
MAX_NEW_TOKENS = 32         # 你的任务只需要输出 Likert 句子，32/64 通常足够
TEMPERATURE = 0.0           # 量表预测建议先用 0，稳定可复现
TOP_P = 1.0
INFER_BATCH_SIZE = 100        # 3090 上先用 1；显存充足再调大
NUM_REPEATS = 1
MIN_FREE_GPU_MEMORY_GB = 6.0

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_NAME_OR_PATH)
print("Train file:", TRAIN_FILE)
print("Test file:", TEST_FILE)
print("Processed train:", PROCESSED_TRAIN_FILE)
print("Processed test:", PROCESSED_TEST_FILE)
print("Model exists:", Path(MODEL_NAME_OR_PATH).exists())
print("Train exists:", TRAIN_FILE.exists())
print("Test exists:", TEST_FILE.exists())


Repo root: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version
Model path: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/models/Qwen3-4B-Instruct-2507
Train file: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/split_data/toy_train.json
Test file: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/split_data/toy_train.json
Processed train: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/processed_data/processed_train_messages.json
Processed test: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/processed_data/processed_test_messages.json
Model exists: True
Train exists: True
Test exists: True


## 2. 导入依赖并检查 CUDA

In [ ]:
import inspect
import json
import re
from typing import Any

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

from llm_lab.data import _apply_chat_template, load_sft_dataset, load_grpo_dataset
from llm_lab.model_utils import ensure_pad_token
from llm_lab.train_utils import ResponseOnlyDataCollator, build_sft_trainer, get_training_args



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/chenyida/miniconda3/envs/unsloth/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. 加载模型并注入 LoRA

这部分基本等同于 Unsloth 官方文档里的核心代码。

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME_OR_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

def disable_thinking_by_default(tokenizer, enable_thinking: bool = False):
    """Make Qwen3 chat templates default to enable_thinking=False.
    """
    apply_chat_template = getattr(tokenizer, "apply_chat_template", None)
    if apply_chat_template is None:
        return tokenizer
    try:
        parameters = inspect.signature(apply_chat_template).parameters
    except (TypeError, ValueError):
        return tokenizer
    if "enable_thinking" not in parameters:
        return tokenizer

    def apply_chat_template_without_thinking(*args, **kwargs):
        kwargs.setdefault("enable_thinking", enable_thinking)
        return apply_chat_template(*args, **kwargs)

    tokenizer.apply_chat_template = apply_chat_template_without_thinking
    return tokenizer


ensure_pad_token(tokenizer)
tokenizer = disable_thinking_by_default(tokenizer, ENABLE_THINKING)
tokenizer.padding_side = "left"
print(f"Qwen thinking enabled: {ENABLE_THINKING}")

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

==((====))==  Unsloth 2025.7.7: Fast Qwen3 patching. Transformers: 4.53.3.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.691 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.32s/it]


Qwen thinking enabled: False


Unsloth 2025.7.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 4. 读取并标准化 train/test 两个 JSON 文件

这里不再依赖 `split` 字段。Notebook 会把原始字段数据转成标准 `messages` 格式：

- train：`system + user(prompt) + assistant(groundtruth)`，用于 SFT。
- test：`system + user(prompt)`；如果 test 里也有 `groundtruth`，会保留下来用于评估。

如果你的答案字段不叫 `groundtruth`，优先在配置区修改 `RESPONSE_FIELD_CANDIDATES`。


In [5]:
def read_json_or_jsonl(path: str | Path) -> list[dict[str, Any]]:
    path = Path(path)
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        raise ValueError(f"Empty data file: {path}")
    if path.suffix.lower() == ".json" or raw.startswith("["):
        rows = json.loads(raw)
    else:
        rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        raise ValueError("Data must be a JSON array or JSONL of objects.")
    return rows


def first_text_field(row: dict[str, Any], candidates: list[str]) -> tuple[str | None, str | None]:
    for field in candidates:
        value = row.get(field)
        if isinstance(value, str) and value.strip():
            return field, value.strip()
    return None, None


def compact_row(row: dict[str, Any], require_response: bool) -> dict[str, Any]:
    prompt = row.get(PROMPT_FIELD)
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError(f"missing non-empty prompt field {PROMPT_FIELD!r}")

    response_field, response = first_text_field(row, RESPONSE_FIELD_CANDIDATES)
    if require_response and response is None:
        raise ValueError(
            "missing answer field; tried "
            f"{RESPONSE_FIELD_CANDIDATES}. Add your real answer field to RESPONSE_FIELD_CANDIDATES."
        )

    compact: dict[str, Any] = {}
    for field in PROCESSED_KEEP_FIELDS:
        if field == RESPONSE_FIELD:
            if response is not None:
                compact[RESPONSE_FIELD] = response
        elif field in row:
            compact[field] = row[field]

    # 保证训练和推理最关键字段一定存在，并统一答案字段名为 groundtruth。
    compact[PROMPT_FIELD] = prompt.strip()
    if response is not None:
        compact[RESPONSE_FIELD] = response
    return compact


def standardize_rows(rows: list[dict[str, Any]], name: str, require_response: bool) -> list[dict[str, Any]]:
    if not rows:
        raise ValueError(f"{name} is empty")
    converted = []
    for idx, row in enumerate(rows):
        try:
            converted.append(compact_row(row, require_response=require_response))
        except ValueError as exc:
            raise ValueError(f"{name} row {idx} cannot be converted: {exc}") from exc
    return converted


train_raw_rows = read_json_or_jsonl(TRAIN_FILE)
test_raw_rows = read_json_or_jsonl(TEST_FILE) if TEST_FILE.exists() else []
train_rows = standardize_rows(train_raw_rows, "train", require_response=True)
test_rows = standardize_rows(test_raw_rows, "test", require_response=False) if test_raw_rows else []

PROCESSED_TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_TEST_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_TRAIN_FILE.write_text(json.dumps(train_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
PROCESSED_TEST_FILE.write_text(json.dumps(test_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"Train rows: {len(train_rows)} from {TRAIN_FILE}")
print(f"Test rows : {len(test_rows)} from {TEST_FILE}")
print(f"Wrote compact processed train data to: {PROCESSED_TRAIN_FILE}")
print(f"Wrote compact processed test data to : {PROCESSED_TEST_FILE}")
print("Train columns:", sorted(train_rows[0].keys()))
if test_rows:
    print("Test columns:", sorted(test_rows[0].keys()))
print("Example processed train row:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:1200])


Train rows: 128 from /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/split_data/toy_train.json
Test rows : 128 from /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/split_data/toy_train.json
Wrote compact processed train data to: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/processed_data/processed_train_messages.json
Wrote compact processed test data to : /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/data/processed_data/processed_test_messages.json
Train columns: ['claim_id', 'condition', 'groundtruth', 'prompt', 'prompt_id']
Test columns: ['claim_id', 'condition', 'groundtruth', 'prompt', 'prompt_id']
Example processed train row:
{
  "prompt_id": "claim001_highInv_highExpert_StrongArg",
  "claim_id": 1,
  "prompt": "You are a cinema customer who may use assisted-listening headphones during screenings. A local cinema chain is collecting audience opinions on accessibility equipment rental and maintenance. One proposal in the survey states that cinemas

## 5. 构造 SFT Dataset

这里训练不直接吃原始 JSON，而是吃上一步写出的 `PROCESSED_TRAIN_FILE`：

- processed train/test 只保留关键字段：`prompt_id`、`claim_id`、`prompt`、`groundtruth`、`condition`。
- `prompt` 会作为 user 输入。
- `groundtruth` 会作为 assistant answer。
- `RESPONSE_ONLY_LOSS=True` 时，loss 只算 assistant answer 部分，不会训练 prompt。


In [ ]:
# 上面已经 monkeypatch tokenizer.apply_chat_template 默认关闭 thinking。
train_dataset = load_sft_dataset(
    PROCESSED_TRAIN_FILE,
    tokenizer,
    response_only_loss = RESPONSE_ONLY_LOSS,
)

print(train_dataset)
print("Columns:", train_dataset.column_names)
print("First row keys:", train_dataset[0].keys())


Dataset({
    features: ['text', 'prompt_text'],
    num_rows: 128
})
Columns: ['text', 'prompt_text']
First row keys: dict_keys(['text', 'prompt_text'])


In [7]:
train_dataset[0]["text"]

'<|im_start|>user\nYou are a cinema customer who may use assisted-listening headphones during screenings. A local cinema chain is collecting audience opinions on accessibility equipment rental and maintenance. One proposal in the survey states that cinemas should charge a small fee for repeated damage to rented assisted-listening headphones.\n\nThe survey result will directly affect the cinema you often visit. If the proposal is supported, customers at that cinema will pay a small fee for repeated damage to rented assisted-listening headphones starting next month.\n\nThe proposal and supporting reasons were provided by experts in the relevant field. These experts have long studied issues related to this topic and have professional experience.\n\nProposal:\nCinemas should charge a small fee for repeated damage to rented assisted-listening headphones.\n\nSupporting reasons:\n1. This policy would help keep assisted-listening headphones available for customers who genuinely need them durin

## 6. SFT 训练

- `RESPONSE_ONLY_LOSS=True`：使用 `Trainer + ResponseOnlyDataCollator`，只对 `groundtruth` 计算 loss。
- `RESPONSE_ONLY_LOSS=False`：使用 TRL `SFTTrainer`，对完整 prompt+answer 计算 loss。

In [8]:
from transformers import Trainer

fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

training_args = get_training_args(
    output_dir = str(SFT_OUTPUT_DIR),
    max_length = MAX_SEQ_LENGTH,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    fp16 = fp16,
    bf16 = bf16,
)

if RESPONSE_ONLY_LOSS:
    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = train_dataset,
        data_collator = ResponseOnlyDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH),
    )
else:
    trainer = build_sft_trainer(model, tokenizer, train_dataset, training_args)

trainer.train()

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 128 | Num Epochs = 8 | Total steps = 32
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.457200
2,0.201100
3,0.266500
4,0.102900
5,0.976000
6,1.538500
7,1.057700
8,0.719100
9,0.265900
10,0.104700


TrainOutput(global_step=32, training_loss=0.2222952269949019, metrics={'train_runtime': 297.6094, 'train_samples_per_second': 3.441, 'train_steps_per_second': 0.108, 'total_flos': 7227722406789120.0, 'train_loss': 0.2222952269949019, 'epoch': 8.0})

## 7. 保存 LoRA adapter / 合并模型

In [9]:
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))
print("Saved LoRA adapter to:", SFT_OUTPUT_DIR)

# 如果需要部署合并模型，取消下面一行注释：
# trainer.model.save_pretrained_merged(str(SFT_OUTPUT_DIR) + "_merged_16bit", tokenizer, save_method="merged_16bit")

Saved LoRA adapter to: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/outputs/lora_files/qwen3_4b_unsloth_lora


## 8. 单条推理 sanity check

从测试集拿一条 prompt 看模型输出格式是否正确。

In [11]:
def generation_prompt_from_row(row: dict[str, Any]) -> str:
    messages = row.get("messages")
    if isinstance(messages, list) and messages:
        prompt_messages = [dict(message) for message in messages]
        if prompt_messages and str(prompt_messages[-1].get("role", "")).strip().lower() in {"assistant", "respondent"}:
            prompt_messages = prompt_messages[:-1]
    else:
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[PROMPT_FIELD]},
        ]
    # 不直接传 enable_thinking，兼容旧版 _apply_chat_template。
    # tokenizer.apply_chat_template 已默认关闭 thinking。
    return _apply_chat_template(tokenizer, prompt_messages, add_generation_prompt=True)


def generate_one(row: dict[str, Any]) -> str:
    text = generation_prompt_from_row(row)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    do_sample = TEMPERATURE > 0
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": do_sample,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
    outputs = trainer.model.generate(**inputs, **generation_kwargs)
    new_tokens = outputs[:, inputs.input_ids.shape[-1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

sample = test_rows[0] if test_rows else train_rows[0]
print("Prompt id:", sample.get(ID_FIELD))
print("Groundtruth:", sample.get(RESPONSE_FIELD))
print("Model output:", generate_one(sample))


Prompt id: claim001_highInv_highExpert_StrongArg
Groundtruth: My attitude score toward this proposal is: 8
Model output: My attitude score toward this proposal is: 8


## 9. 测试集批量推理并保存结果

输出文件会保留你的原始字段（如 `prompt_id`、`claim`、`condition`、`groundtruth` 等），并新增：

- `model_output`：模型完整输出。
- `parsed_score`：从输出中抽取的 1-11 分数；解析失败则为 `None`。
- `is_exact_match`：模型输出和 `groundtruth` 去空格后是否完全一致。

In [13]:
def left_pad_tokenize(tokenizer, texts: list[str]) -> dict[str, torch.Tensor]:
    """Manually left-pad pre-rendered prompts for decoder-only generation."""
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    if pad_token_id is None:
        raise ValueError("Tokenizer must define pad_token_id or eos_token_id for batch inference padding.")
    encoded = [tokenizer(text, add_special_tokens=False, return_attention_mask=True) for text in texts]
    max_len = max(len(item["input_ids"]) for item in encoded)
    input_ids, attention_mask = [], []
    for item in encoded:
        ids = item["input_ids"]
        mask = item.get("attention_mask", [1] * len(ids))
        pad_len = max_len - len(ids)
        input_ids.append([pad_token_id] * pad_len + ids)
        attention_mask.append([0] * pad_len + mask)
    batch = {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }
    return {key: value.to("cuda") for key, value in batch.items()}


def parse_likert_score(text: str) -> int | None:
    # 优先匹配任务指定前缀后的数字；失败再退化为最后一个 1-11 数字。
    prefix_match = re.search(r"My attitude score toward this proposal is:\s*(\d{1,2})", text, flags=re.I)
    candidates = [prefix_match.group(1)] if prefix_match else re.findall(r"\b(?:1[01]|[1-9])\b", text)
    if not candidates:
        return None
    score = int(candidates[-1])
    return score if 1 <= score <= 11 else None


def batch_generate(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if not rows:
        return []
    results = [{field: row[field] for field in PROCESSED_KEEP_FIELDS if field in row} for row in rows]
    pending: list[tuple[int, str]] = []
    for row_idx, row in enumerate(rows):
        text = generation_prompt_from_row(row)
        for _ in range(NUM_REPEATS):
            pending.append((row_idx, text))

    outputs_by_row = [[] for _ in rows]
    do_sample = TEMPERATURE > 0
    for start in range(0, len(pending), INFER_BATCH_SIZE):
        batch = pending[start:start + INFER_BATCH_SIZE]
        row_ids = [row_id for row_id, _ in batch]
        texts = [text for _, text in batch]
        inputs = left_pad_tokenize(tokenizer, texts)
        generation_kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": do_sample,
            "use_cache": True,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        }
        if do_sample:
            generation_kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
        outputs = trainer.model.generate(**inputs, **generation_kwargs)
        new_tokens = outputs[:, inputs["input_ids"].shape[-1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for row_id, output_text in zip(row_ids, decoded):
            outputs_by_row[row_id].append(output_text.strip())
        print(f"Processed {min(start + len(batch), len(pending))}/{len(pending)} generations")

    for row, outs in zip(results, outputs_by_row):
        output_value = outs if NUM_REPEATS > 1 else outs[0]
        first_output = outs[0] if outs else ""
        row["model_output"] = output_value
        row["parsed_score"] = parse_likert_score(first_output)
        row["is_exact_match"] = first_output.strip() == str(row.get(RESPONSE_FIELD, "")).strip()
    return results


pred_rows = batch_generate(test_rows)
PRED_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
PRED_OUTPUT_FILE.write_text(json.dumps(pred_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {len(pred_rows)} predictions to {PRED_OUTPUT_FILE}")
if pred_rows:
    print(json.dumps({k: pred_rows[0].get(k) for k in [ID_FIELD, RESPONSE_FIELD, "model_output", "parsed_score", "is_exact_match"]}, ensure_ascii=False, indent=2))


Processed 100/128 generations


/home/chenyida/miniconda3/envs/unsloth/lib/python3.10/site-packages/unsloth/kernels/utils.py:661: UserWarning: An output with one or more elements was resized since it had shape [1, 28, 2560], which does not match the required output shape [28, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:30.)
  out = torch_matmul(X, W, out = out)


Processed 128/128 generations
Wrote 128 predictions to /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/outputs/predictions/qwen3_4b_test_predictions.json
{
  "prompt_id": "claim001_highInv_highExpert_StrongArg",
  "groundtruth": "My attitude score toward this proposal is: 8",
  "model_output": "My attitude score toward this proposal is: 8",
  "parsed_score": 8,
  "is_exact_match": true
}


In [14]:
import json
import re
import csv
from pathlib import Path
from statistics import mean


pred_path = Path(PRED_OUTPUT_FILE)

ELM_STATS_FILE = pred_path.with_name(pred_path.stem + "_elm_stats.json")
ELM_CLAIM_STATS_CSV = pred_path.with_name(pred_path.stem + "_claim_stats.csv")


score_re = re.compile(
    r"My attitude score toward this proposal is:\s*(11|10|[1-9])",
    re.IGNORECASE,
)


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def normalize_score(score: int) -> float:
    """
    11-point Likert scale:
    1  -> 0.0
    11 -> 1.0
    """
    return (score - 1) / 10


def extract_score_from_text(text: str):
    if not isinstance(text, str):
        return None

    m = score_re.search(text)
    if m:
        return int(m.group(1))

    return None


def get_pred_score(item: dict):
    """
    优先使用第 9 步已经解析好的 parsed_score。
    如果没有 parsed_score，再从 model_output 中用正则解析。
    """
    parsed_score = item.get("parsed_score", None)

    if isinstance(parsed_score, int) and 1 <= parsed_score <= 11:
        return parsed_score

    if isinstance(parsed_score, float) and parsed_score.is_integer() and 1 <= int(parsed_score) <= 11:
        return int(parsed_score)

    model_output = item.get("model_output", "")
    score = extract_score_from_text(model_output)

    if score is not None:
        return score

    return None


def parse_claim_id(item: dict):
    """
    优先读取 claim_id。
    如果 prediction 文件里没有 claim_id，则从 prompt_id 解析：
    claim001_highInv_highExpert_StrongArg -> 1
    """
    if "claim_id" in item and item["claim_id"] is not None:
        return int(item["claim_id"])

    prompt_id = item.get("prompt_id", "")
    m = re.search(r"claim0*(\d+)", prompt_id, re.IGNORECASE)

    if not m:
        raise ValueError(f"Cannot parse claim_id from prompt_id: {prompt_id}")

    return int(m.group(1))


def parse_condition_key(item: dict):
    """
    返回：
    (involvement, source_expertise, argument_quality)

    例如：
    ("high", "high", "strong")
    """
    cond = item.get("condition", None)

    if isinstance(cond, dict):
        involvement = cond.get("involvement")
        source_expertise = cond.get("source_expertise")
        argument_quality = cond.get("argument_quality")

        if involvement and source_expertise and argument_quality:
            return (
                involvement.lower(),
                source_expertise.lower(),
                argument_quality.lower(),
            )

    # 如果 prediction 文件里没有 condition，则从 prompt_id 解析
    prompt_id = item.get("prompt_id", "")

    inv_m = re.search(r"(high|low)Inv", prompt_id, re.IGNORECASE)
    src_m = re.search(r"(high|low)Expert", prompt_id, re.IGNORECASE)
    arg_m = re.search(r"(strong|weak)Arg", prompt_id, re.IGNORECASE)

    if not (inv_m and src_m and arg_m):
        raise ValueError(f"Cannot parse condition from prompt_id: {prompt_id}")

    involvement = inv_m.group(1).lower()
    source_expertise = src_m.group(1).lower()
    argument_quality = arg_m.group(1).lower()

    return involvement, source_expertise, argument_quality


def mean_or_none(values):
    return mean(values) if values else None


required_condition_keys = [
    ("high", "high", "strong"),
    ("high", "high", "weak"),
    ("high", "low", "strong"),
    ("high", "low", "weak"),
    ("low", "high", "strong"),
    ("low", "high", "weak"),
    ("low", "low", "strong"),
    ("low", "low", "weak"),
]


data = load_json(pred_path)

# claim_id -> condition_key -> list[normalized_score]
score_groups = {}
invalid_items = []

for item in data:
    try:
        claim_id = parse_claim_id(item)
        condition_key = parse_condition_key(item)
        pred_score = get_pred_score(item)

        if pred_score is None:
            invalid_items.append({
                "prompt_id": item.get("prompt_id"),
                "reason": "Cannot parse prediction score",
                "model_output": item.get("model_output"),
            })
            continue

        normalized = normalize_score(pred_score)

        score_groups.setdefault(claim_id, {})
        score_groups[claim_id].setdefault(condition_key, [])
        score_groups[claim_id][condition_key].append(normalized)

    except Exception as e:
        invalid_items.append({
            "prompt_id": item.get("prompt_id"),
            "reason": str(e),
            "model_output": item.get("model_output"),
        })


claim_stats = []
incomplete_claims = []

for claim_id in sorted(score_groups.keys()):
    cond_scores = score_groups[claim_id]

    missing_keys = [
        key for key in required_condition_keys
        if key not in cond_scores or len(cond_scores[key]) == 0
    ]

    if missing_keys:
        incomplete_claims.append({
            "claim_id": claim_id,
            "missing_conditions": [list(x) for x in missing_keys],
        })
        continue

    # 每个 condition 如果有多次输出，则先求均值
    s = {
        key: mean(cond_scores[key])
        for key in required_condition_keys
    }

    HHs = s[("high", "high", "strong")]
    HHw = s[("high", "high", "weak")]
    HLs = s[("high", "low", "strong")]
    HLw = s[("high", "low", "weak")]

    LHs = s[("low", "high", "strong")]
    LHw = s[("low", "high", "weak")]
    LLs = s[("low", "low", "strong")]
    LLw = s[("low", "low", "weak")]

    # 高涉入下，应该更看 argument quality
    D_H_Arg = mean([
        HHs - HHw,
        HLs - HLw,
    ])

    # 低涉入下，argument quality 的影响应该更弱
    D_L_Arg = mean([
        LHs - LHw,
        LLs - LLw,
    ])

    # ELM argument sensitivity:
    # 高涉入的强弱论据差异 - 低涉入的强弱论据差异
    Delta_Arg = D_H_Arg - D_L_Arg

    # 高涉入下，source expertise 的影响应该较弱
    D_H_Src = mean([
        HHs - HLs,
        HHw - HLw,
    ])

    # 低涉入下，source expertise 的影响应该更强
    D_L_Src = mean([
        LHs - LLs,
        LHw - LLw,
    ])

    # ELM source cue sensitivity:
    # 低涉入的来源专业性差异 - 高涉入的来源专业性差异
    Delta_Src = D_L_Src - D_H_Src

    claim_stat = {
        "claim_id": claim_id,

        "HHs": HHs,
        "HHw": HHw,
        "HLs": HLs,
        "HLw": HLw,
        "LHs": LHs,
        "LHw": LHw,
        "LLs": LLs,
        "LLw": LLw,

        "D_H_Arg": D_H_Arg,
        "D_L_Arg": D_L_Arg,
        "Delta_Arg": Delta_Arg,

        "D_H_Src": D_H_Src,
        "D_L_Src": D_L_Src,
        "Delta_Src": Delta_Src,

        "Arg_ELM_pass": Delta_Arg > 0,
        "Src_ELM_pass": Delta_Src > 0,
        "Both_ELM_pass": (Delta_Arg > 0 and Delta_Src > 0),
    }

    claim_stats.append(claim_stat)


if len(claim_stats) == 0:
    raise RuntimeError(
        "No complete claims found. Please check whether the prediction file contains all 8 conditions for each claim."
    )


summary = {
    "prediction_file": str(pred_path),
    "num_prediction_items": len(data),
    "num_valid_prediction_items": len(data) - len(invalid_items),
    "num_invalid_prediction_items": len(invalid_items),

    "num_complete_claims": len(claim_stats),
    "num_incomplete_claims": len(incomplete_claims),

    "D_H_Arg": mean(x["D_H_Arg"] for x in claim_stats),
    "D_L_Arg": mean(x["D_L_Arg"] for x in claim_stats),
    "Delta_Arg": mean(x["Delta_Arg"] for x in claim_stats),

    "D_H_Src": mean(x["D_H_Src"] for x in claim_stats),
    "D_L_Src": mean(x["D_L_Src"] for x in claim_stats),
    "Delta_Src": mean(x["Delta_Src"] for x in claim_stats),

    "Arg_ELM_pass_rate": mean(1 if x["Arg_ELM_pass"] else 0 for x in claim_stats),
    "Src_ELM_pass_rate": mean(1 if x["Src_ELM_pass"] else 0 for x in claim_stats),
    "Both_ELM_pass_rate": mean(1 if x["Both_ELM_pass"] else 0 for x in claim_stats),
}


result = {
    "summary": summary,
    "claim_stats": claim_stats,
    "incomplete_claims": incomplete_claims,
    "invalid_items": invalid_items,
}


save_json(result, ELM_STATS_FILE)


# 同时导出一个 claim-level CSV，方便后续看表
with open(ELM_CLAIM_STATS_CSV, "w", encoding="utf-8", newline="") as f:
    fieldnames = list(claim_stats[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(claim_stats)


print("ELM statistics saved to:", ELM_STATS_FILE)
print("Claim-level CSV saved to:", ELM_CLAIM_STATS_CSV)
print()
print(json.dumps(summary, ensure_ascii=False, indent=2))

ELM statistics saved to: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/outputs/predictions/qwen3_4b_test_predictions_elm_stats.json
Claim-level CSV saved to: /nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/outputs/predictions/qwen3_4b_test_predictions_claim_stats.csv

{
  "prediction_file": "/nfs/users/chenyida/MyCode/LLMs_ELM_unsloth-version/outputs/predictions/qwen3_4b_test_predictions.json",
  "num_prediction_items": 128,
  "num_valid_prediction_items": 128,
  "num_invalid_prediction_items": 0,
  "num_complete_claims": 16,
  "num_incomplete_claims": 0,
  "D_H_Arg": 0.403125,
  "D_L_Arg": 0.0,
  "Delta_Arg": 0.403125,
  "D_H_Src": 0.10937500000000001,
  "D_L_Src": 0.3375,
  "Delta_Src": 0.22812500000000002,
  "Arg_ELM_pass_rate": 1,
  "Src_ELM_pass_rate": 1,
  "Both_ELM_pass_rate": 1
}


## 10. 可选：GRPO / RL

默认不运行。你要做 RL 时再取消注释，并把 reward 函数替换成你的任务 reward。例如你的任务可以用 `parsed_score` 与目标分数的距离设计 reward。

```python
# from unsloth import PatchFastRL
# PatchFastRL("grpo", FastLanguageModel)
# from trl import GRPOConfig, GRPOTrainer
# rl_dataset = load_grpo_dataset(PROCESSED_TRAIN_FILE, answer_field=RESPONSE_FIELD)
# ...
```


## 11. 对应脚本命令

Notebook 会先把原始数据写成 `data/processed_data/processed_train_messages.json` 和 `data/processed_data/processed_test_messages.json`。Notebook 跑通后，正式训练可以直接用 processed 文件：

```bash
CUDA_VISIBLE_DEVICES=2 python scripts/train_lora.py \
  --model_name_or_path models/Qwen3-1.7B \
  --train_file data/processed_data/processed_train_messages.json \
  --output_dir outputs/lora_files/qwen3_1.7b_unsloth_lora

CUDA_VISIBLE_DEVICES=2 python scripts/batch_infer_lora.py \
  --model_name_or_path outputs/lora_files/qwen3_1.7b_unsloth_lora \
  --input_file data/processed_data/processed_test_messages.json \
  --output_file outputs/predictions/qwen3_1.7b_test_predictions.json \
  --overwrite
```
